# Bài 14 · Kể chuyện bằng dữ liệu và thẩm định bản phân tích do AI viết

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Phân biệt phân tích để khám phá và trình bày để giải thích; nêu ý chính của một phân tích trong một câu.
2. Dựng câu chuyện từ dữ liệu: kết luận trước, tiêu đề nêu kết luận, con số đặt trong phép so sánh, giới hạn nêu cạnh kết luận.
3. Nhận ra các lỗi thường gặp khi diễn giải: tương quan và nhân quả, cơ cấu nhóm, mốc so sánh, số liệu thay đổi theo mốc chụp, lời diễn đạt quá mức.
4. Thẩm định (audit) một bản phân tích do AI viết: tính lại số liệu, kiểm tra phương pháp, kiểm tra diễn giải, kết luận giữ, sửa hay chưa đủ căn cứ.

**Tài liệu:** Wilke, *Fundamentals of Data Visualization*,
[chương 29](https://clauswilke.com/dataviz/telling-a-story.html) và [chương 22](https://clauswilke.com/dataviz/figure-titles-captions.html);
[Calling Bullshit](https://callingbullshit.org/syllabus.html) (Bergstrom và West).
Dữ liệu: Santiago, mốc chụp **29/06/2026** (giá là **CLP/đêm**) và bảng đánh giá ở mốc chụp **27/09/2025**.
Báo cáo AI trong bài là output thật của `gemini-3.5-flash-lite` ngày 02/11/2026; notebook lưu sẵn nguyên văn,
nên **không có khoá API vẫn chạy hết**. Các ví dụ được kiểm tra bằng pandas **3.0.3**.

## Chuẩn bị

Ô cài đặt dành cho Colab. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.
Chạy các ô demo theo thứ tự; ô **Bài tập** để bạn tự viết lời giải và nhận xét.
Ô dưới dùng file đã tải nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn (khoảng 35 MB).

In [ ]:
%pip -q install pandas==3.0.3 "google-genai>=2.19"

In [ ]:
import json, os
from pathlib import Path

import pandas as pd


def doc(ten_tep, url, **kw):
    """Đọc file đã tải trong data/ nếu có, nếu không thì đọc URL nguồn."""
    tep = Path("data") / ten_tep
    return pd.read_csv(tep if tep.exists() else url, **kw)


GOC = "https://data.insideairbnb.com/chile/rm/santiago"
df = doc("listings_santiago_2026-06-29.csv", f"{GOC}/2026-06-29/visualisations/listings.csv")
rv = doc("reviews_santiago_2026-06-29.csv", f"{GOC}/2026-06-29/visualisations/reviews.csv",
         parse_dates=["date"])
rv = rv[rv["date"] <= "2026-06-29"]                  # cắt theo mốc chụp (Bài 8)
print(df.shape, rv.shape)
print("pandas:", pd.__version__)

## 1. Khám phá và trình bày

Phân tích để **khám phá** (cho chính mình, nhiều hình) khác với **trình bày** (cho người đọc, vài hình chọn lọc,
tiêu đề nêu kết luận).

Trước khi viết, trả lời: ai đọc, họ cần quyết định gì, họ cần thấy gì để quyết định.

**Ý chính** (big idea) là một câu hoàn chỉnh nêu phát hiện chính, khác với chủ đề (chỉ gọi tên điều được bàn).
Viết ý chính trước khi soạn báo cáo để chọn hình và số liệu cần đưa vào. Trong báo cáo, ý chính là câu mở đầu.

### Bài tập 1: Viết ý chính trong một câu

Bảng dưới là giá trung vị (CLP/đêm), số chỗ ở và tỷ lệ (%) chỗ ở của thành phố ở các quận có từ 500 chỗ ở.

**Sở du lịch muốn biết nên ưu tiên quản lý cho thuê ngắn hạn ở quận nào. Viết ý chính trong một câu từ bảng này.**

In [ ]:
tk = df.groupby("neighbourhood")["price"].agg(
    median="median", n="size")
tk["% chỗ ở"] = (tk["n"] / len(df) * 100).round(1)   # trên 18.534 chỗ ở
print(tk[tk["n"] >= 500].sort_values("median"))

**Gợi ý lời giải.** Người đọc là sở du lịch, câu hỏi là nên quản lý ở đâu, nên dùng cột số chỗ ở và tỷ lệ.
Câu mẫu: *"Gần 40% chỗ ở Airbnb của thành phố nằm ở quận Santiago, gấp 2,4 lần quận đứng thứ hai."*
Câu về giá Lo Barnechea gây chú ý nhưng không trả lời câu hỏi của người đọc.

**Ý chính của bạn:** …

## 2. Dựng câu chuyện từ dữ liệu

Một câu chuyện dữ liệu có bốn phần: **mở đầu** (bối cảnh), **vấn đề** (câu hỏi), **phân tích**, **kết luận**.

Đặt kết luận lên trước, bằng chứng sau, cách làm cuối cùng (hoặc ở phụ lục).

Số đánh giá mỗi tháng từ 01/2024 đến 05/2026 (bỏ tháng 6/2026 chưa trọn vẹn, cách đếm như Bài 8 và Bài 12), rồi cùng một hình với tiêu đề gọi tên và tiêu đề nêu kết luận:

In [ ]:
thang = (rv.set_index("date").resample("ME").size()
         .loc["2024":"2026-05"])        # bỏ tháng 6/2026 chưa trọn vẹn
print(pd.concat([thang["2025-05"], thang["2026-05"]]))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
axes[0].plot(thang.index, thang.values)
axes[0].set_title("Số đánh giá theo tháng")                 # gọi tên

axes[1].plot(thang.index, thang.values, color="#b5b5b5", lw=2)
for thang_so in ["2025-05", "2026-05"]:
    d = thang.loc[thang_so].index[0]
    axes[1].plot(d, thang[d], "o", color="#E8890C")
    axes[1].annotate(f"{thang[d]:,}".replace(",", "."), (d, thang[d]),
                     xytext=(0, -20), textcoords="offset points", ha="center", color="#E8890C")
tang = thang["2026-05"].iloc[0] / thang["2025-05"].iloc[0] - 1
axes[1].set_title(f"Số đánh giá tháng 5/2026 tăng {tang:.0%} so với tháng 5/2025",
                  loc="left", fontweight="bold")
for ax in axes:
    ax.set_xlabel("Tháng")
axes[0].set_ylabel("Số đánh giá")   # sharey: hai hình chung trục y
plt.tight_layout()
plt.show()

Bảng giữ con số chính xác; câu văn nêu phép so sánh (cao hơn bao nhiêu phần trăm, gấp mấy lần, 1 trên mấy).

Nêu giới hạn ngay cạnh kết luận, không dồn hết vào cuối báo cáo.

| Giới hạn | Bài | Cách nêu trong báo cáo |
|---|---|---|
| Số đánh giá không phải số lượt thuê | 8 | "số đánh giá", không viết "lượt khách" |
| Tháng cuối chưa trọn vẹn | 8 | ghi "đến 29/06" hoặc bỏ tháng đó |
| 846 chỗ ở thiếu giá | 10 | "trên 17.688 chỗ ở có giá" |
| Nhãn LLM đo trên 24 đánh giá | 11 | "F1 0,92 trên mẫu nhỏ" |
| Thiếu đánh giá của chỗ ở đã ngừng đăng | Lab 13 | "mức tăng thực tế có thể thấp hơn" |

### Bài tập 2: Tiêu đề và hai câu diễn giải

Lấy hình "So sánh trên cùng thang đo" của Bài 12 (giá nguyên căn và phòng riêng, thang log10).
Viết tiêu đề nêu kết luận và hai câu diễn giải: một câu kết luận, một câu giới hạn.

**Gợi ý lời giải.** Trên cùng thang log, phòng riêng nằm bên trái nguyên căn; hai phân phối chồng lên nhau một phần.
Thang log khó đọc số chính xác, nên tính giá trung vị trong notebook.

- *Tiêu đề:* Giá phòng riêng chỉ bằng khoảng một nửa giá nguyên căn
- *Kết luận:* Giá trung vị của phòng riêng là 34.235 CLP/đêm, của nguyên căn là 64.900 CLP/đêm.
- *Giới hạn:* Giá hai loại trùng nhau khá nhiều: 14% phòng riêng đắt hơn giá trung vị của nguyên căn.

**Tiêu đề và hai câu diễn giải của bạn:** …

## 3. Lỗi thường gặp khi diễn giải số liệu

Chỗ ở có chữ "metro" trong tên có giá trung vị thấp hơn. Có thể kết luận gần metro làm giá rẻ hơn không?

In [ ]:
co_metro = df["name"].str.contains("metro", case=False, na=False)
nhom = co_metro.map({True: 'Có "metro"',
                     False: 'Không có "metro"'}).rename("nhóm")
gia_tv = df.groupby(nhom)["price"].median()
print(gia_tv)
ti_le = gia_tv['Có "metro"'] / gia_tv['Không có "metro"'] - 1
print(f"chênh lệch: {ti_le:.1%}")

Hai nhóm phân bố khác nhau giữa các quận. Bảng dưới là giá trung vị của năm quận có từ 500 chỗ ở,
và tỷ lệ (%) chỗ ở của mỗi nhóm nằm trong từng quận. Sau đó so sánh giá mỗi chỗ ở với giá trung vị của chính quận đó.

In [ ]:
quan_dong = ["Santiago", "Ñuñoa", "Providencia", "Las Condes",
             "Lo Barnechea"]                 # năm quận có từ 500 chỗ ở
gia_quan = df.groupby("neighbourhood")["price"].median()
ti_trong = pd.crosstab(df["neighbourhood"], nhom, normalize="columns")
bang = (ti_trong.loc[quan_dong] * 100).round(1)  # % chỗ ở của mỗi nhóm
bang.insert(0, "giá trung vị quận", gia_quan[quan_dong])
print(bang)

In [ ]:
# giá mỗi chỗ ở so với giá trung vị của chính quận đó (1 = bằng nhau)
so_voi_quan = df["price"] / df["neighbourhood"].map(gia_quan)
print(so_voi_quan.groupby(nhom).median().round(3))

**Nghịch lý Simpson** (số minh hoạ): từng nhóm đều tăng nhưng con số gộp lại giảm, vì tỷ trọng các nhóm thay đổi.
Dữ liệu Santiago giữa hai mốc chụp không có nghịch lý này (ô dưới).

Cùng một dữ liệu, chọn mốc so sánh khác nhau cho các con số rất khác nhau:

In [ ]:
thang_tron = (rv.set_index("date").resample("ME").size()
              .loc[:"2026-05"])
t5 = thang_tron["2026-05"].iloc[0]
nam = thang_tron.groupby(thang_tron.index.year).sum()
goc_2020 = thang_tron["2020-04"].iloc[0]       # đáy COVID-19
tb_2019 = thang_tron["2019"].mean()
print("05/2026 / 04/2020:", round(t5 / goc_2020, 1))
print("05/2026 / TB tháng 2019:", round(t5 / tb_2019, 1))
print("2025 / 2020:", round(nam[2025] / nam[2020], 1))
print("2025 / 2019:", round(nam[2025] / nam[2019], 1))

Đếm cùng các tháng ở hai mốc chụp khác nhau:

In [ ]:
url9 = f"{GOC}/2025-09-27/visualisations/reviews.csv"
rv9 = doc("reviews_santiago_2025-09-27.csv", url9,
          parse_dates=["date"])
rv9 = rv9[rv9["date"] <= "2025-09-27"]
hai_moc = pd.DataFrame({
    "mốc 27/09/2025": rv9.set_index("date").resample("ME").size(),
    "mốc 29/06/2026": rv.set_index("date").resample("ME").size(),
}).loc["2025-06":"2025-09"].astype(int)
print(hai_moc)

Chọn từ theo độ lớn của số liệu và cỡ mẫu (ví dụ minh hoạ):

| Số liệu | Diễn đạt quá mức | Diễn đạt đúng mức |
|---|---|---|
| +3% so với cùng tháng năm trước | "bùng nổ" | "tăng nhẹ" |
| hệ số tương quan 0,2 | "liên hệ chặt chẽ" | "liên hệ yếu" |
| n = 30 | "xu hướng rõ rệt" | "mẫu còn nhỏ, chưa đủ để kết luận" |

## 4. Thẩm định một bản phân tích do AI viết

Bảng tóm tắt số liệu gửi cho mô hình, tính từ dữ liệu thật:

In [ ]:
thang = rv.set_index("date").resample("ME").size()
co_metro = df["name"].str.contains("metro", case=False, na=False)
tk = df.groupby("neighbourhood")["price"].agg(median="median", n="size")
top = tk[tk["n"] >= 500].sort_values("median", ascending=False)

def vn(x, nd=0):
    return f"{x:,.{nd}f}".replace(",", "X").replace(".", ",").replace("X", ".")

TOM_TAT = f"""Dữ liệu Inside Airbnb, Santiago (Chile), mốc chụp 29/06/2026.
- Tổng số chỗ ở: {vn(len(df))}; tỷ lệ nguyên căn: {vn(df['room_type'].eq('Entire home/apt').mean()*100, 1)}%.
- Giá (CLP/đêm), {vn(df['price'].notna().sum())} chỗ ở có giá: trung bình {vn(df['price'].mean())}; trung vị {vn(df['price'].median())}.
- Số đánh giá theo tháng: 03/2026: {vn(thang['2026-03'].iloc[0])}; 04/2026: {vn(thang['2026-04'].iloc[0])}; 05/2026: {vn(thang['2026-05'].iloc[0])}; 06/2026 (đến 29/06): {vn(thang['2026-06'].iloc[0])}. Cùng kỳ 06/2025: {vn(thang['2025-06'].iloc[0])}.
- Giá trung vị chỗ ở có chữ "metro" trong tên: {vn(df.loc[co_metro,'price'].median())} (n = {vn(co_metro.sum())}); các chỗ ở còn lại: {vn(df.loc[~co_metro,'price'].median())}.
- Giá trung vị theo quận (quận có từ 500 chỗ ở): """ + "; ".join(f"{q} {vn(r['median'])} (n = {vn(r['n'])})" for q, r in top.iterrows()) + "."

print(TOM_TAT)

Ô dưới gửi yêu cầu cho Gemini nếu có khoá API (cách lấy khoá ở Bài 11). Không có khoá thì dùng bản lưu từ lần gọi
ngày 02/11/2026. Chạy lại có thể ra báo cáo khác.

In [ ]:
BAO_CAO_LUU_SAN = "Dưới đây là báo cáo thị trường Airbnb ngắn gọn, ấn tượng tại Santiago (Chile) tính đến tháng 06/2026 dành cho nhà đầu tư:\n\n*   **Quy mô lớn và tập trung vào phân khúc nguyên căn:** Thị trường hiện có **18.534** chỗ ở, trong đó loại hình cho thuê nguyên căn áp đảo với tỷ lệ **81,0%**, cho thấy tiềm năng khai thác dòng tiền từ khách du lịch tìm kiếm sự riêng tư.\n*   **Khoảng cách lớn giữa giá trung bình và trung vị phản ánh phân hóa mạnh:** Mức giá trung bình đạt **118.200 CLP/đêm**, nhưng giá trung vị chỉ là **59.000 CLP/đêm** (trên tổng số 17.688 chỗ ở có giá), chứng minh thị trường bị kéo tăng mạnh bởi một lượng nhỏ bất động sản siêu cao cấp.\n*   **Thanh khoản và lượng khách ổn định ngay cả trong mùa thấp điểm:** Số lượng đánh giá tháng 06/2026 đạt **16.112** (tính đến ngày 29/06), tăng trưởng so với mức **15.192** của cùng kỳ năm 2025, cho thấy nhu cầu lưu trú duy trì sức hút tốt bất chấp thời điểm giao mùa.\n*   **Định giá sát nút hệ thống giao thông công cộng:** Các chỗ ở có từ \"metro\" trong tên có giá trung vị **53.926 CLP/đêm** (với **2.959** căn), thấp hơn không đáng kể so với mức **60.478 CLP/đêm** của phần còn lại, khẳng định lợi thế vị trí sát tàu điện nhưng chưa được định giá quá cao.\n*   **Biên độ giá cực kỳ rộng giữa các quận trung tâm và ngoại ô:** Giá trung vị dao động mạnh theo khu vực (từ **500** chỗ ở trở lên), từ mức bình dân **47.755 CLP** tại quận Santiago (**7.182** căn) lên đến phân khúc siêu sang **426.230 CLP** tại Lo Barnechea (**824** căn)."

In [ ]:
PROMPT = ("Viết một báo cáo thị trường Airbnb ngắn, ấn tượng cho nhà đầu tư, gồm 5 gạch đầu dòng. "
          "Mỗi gạch đầu dòng là một nhận định kèm con số, dựa trên số liệu dưới đây.\n\n" + TOM_TAT)

try:
    from google.colab import userdata          # Colab: 🔑 Secrets
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                               # máy cá nhân
    API_KEY = os.environ.get("GEMINI_API_KEY")
CO_KHOA = bool(API_KEY)

if CO_KHOA:
    from google import genai
    client = genai.Client(api_key=API_KEY)
    tra_loi = client.interactions.create(model="gemini-3.5-flash-lite", input=PROMPT, store=False)
    BAO_CAO = tra_loi.output_text
else:
    BAO_CAO = BAO_CAO_LUU_SAN               # bản lưu từ lần gọi ngày 02/11/2026
print(BAO_CAO)

Báo cáo có năm ý; phần dưới thẩm định từng ý.

Bốn bước thẩm định: (1) tính lại số liệu, (2) kiểm tra phương pháp, (3) kiểm tra diễn giải,
(4) kết luận: giữ, sửa hoặc chưa đủ căn cứ.

Ý 2 của báo cáo nói về giá trung bình và trung vị:

In [ ]:
gia = df["price"].dropna()
print(f"trung bình {gia.mean():.0f}, trung vị {gia.median():.0f}")
print(f"P99 {gia.quantile(0.99):.0f}, lớn nhất {gia.max():.0f}")

Ý 3 nói số đánh giá tháng 6/2026 tăng so với tháng 6/2025:

In [ ]:
print(hai_moc.loc["2025-06"])     # tháng 6/2025 ở hai mốc chụp

Ý 4 nói về chỗ ở có chữ "metro" trong tên (xem mục 3):

### Bài tập 3: Thẩm định ý 1 và ý 5

Áp dụng bốn bước thẩm định cho ý 1 và ý 5 của báo cáo (in ở ô trên). Kết luận giữ, sửa hay chưa đủ căn cứ; ghi lý do.

**Thẩm định ý 1 và ý 5 của bạn:** …

**Gợi ý lời giải.** Ý 1: số đúng; "tiềm năng khai thác dòng tiền" là suy diễn vì dữ liệu không có doanh thu. Ý 5: số đúng;
"từ 500 chỗ ở" là điều kiện lọc (5 quận); Lo Barnechea đắt vì phần lớn chỗ ở thuộc vùng núi có khu trượt tuyết.

- *Ý 1 sửa thành:* Santiago có 18.534 chỗ ở trên Airbnb, 81,0% là nguyên căn.
- *Ý 5 sửa thành:* Trong năm quận đông nhất, giá trung vị từ 47.755 CLP/đêm (Santiago) đến 426.230 CLP/đêm (Lo Barnechea, phần lớn chỗ ở thuộc vùng núi có khu trượt tuyết).

Kết quả thẩm định năm ý:

| Ý | Số liệu | Phương pháp | Diễn giải | Kết luận |
|---|---|---|---|---|
| 1. Quy mô, nguyên căn | ✓ | ✓ | ✗ suy diễn dòng tiền | sửa |
| 2. Giá trung bình | ✓ | ✗ thước đo | ✗ "chứng minh" | sửa |
| 3. Đánh giá tháng 6 | ✓ | ✗ tháng chưa trọn vẹn | ✗ "thanh khoản" | chưa đủ căn cứ |
| 4. Metro | ✓ | ✗ chưa tách theo quận | ✗ nhân quả | sửa |
| 5. Theo quận | ✓ | ✓ | ✗ bối cảnh | sửa |

Mọi con số đều đúng, nhưng không ý nào giữ được nguyên văn.

### Đọc thêm

1. [Wilke, Fundamentals of Data Visualization, chương 29](https://clauswilke.com/dataviz/telling-a-story.html), [chương 22](https://clauswilke.com/dataviz/figure-titles-captions.html)
2. Knaflic, *Storytelling with Data* (2015), chương 1 và 7
3. [Calling Bullshit (Bergstrom và West)](https://callingbullshit.org/syllabus.html), [case study Musicians and Mortality](https://callingbullshit.org/case_studies/case_study_musician_mortality.html)
4. [Berkeley Data 100: Visualization II](https://ds100.org/course-notes/visualization-2)

---

## Luyện tập thêm với thành phố của nhóm

Chọn một đoạn trong báo cáo bài tập lớn của nhóm (hoặc nhờ AI viết một đoạn từ bảng chỉ số của nhóm).
Thẩm định đoạn đó bằng bốn bước, ghi kết luận cho từng ý, rồi viết lại đoạn mở đầu: kết luận trước, giới hạn cạnh kết luận.

---

## Tóm tắt bài học

- Khám phá để tìm điều đáng nói; trình bày cho người đọc, nêu ý chính trong một câu.
- Kết luận trước, tiêu đề nêu kết luận, con số đặt trong phép so sánh, giới hạn cạnh kết luận.
- Tránh các lỗi diễn giải thường gặp: suy ra nhân quả, bỏ qua cơ cấu nhóm, chọn mốc so sánh bất thường, quên mốc chụp, nói quá mức.
- Thẩm định: tính lại số liệu, kiểm tra phương pháp, kiểm tra diễn giải, rồi kết luận giữ, sửa hoặc chưa đủ căn cứ.

Hoàn thành ba bài tập rồi chạy **Restart session and run all**.

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

**Bài sau:** trình bày và vấn đáp bài tập lớn (Bài 15).